In [ ]:
from dotenv import load_dotenv
import os 
import numpy as np
from scipy.stats import kruskal, mannwhitneyu

load_dotenv()

# Load base path
base_path = os.environ.get("BASE")
output_path = os.environ.get("OUTPUT")
input_path = os.environ.get("MAIN")

# load helpers 
from utils import norm_entropy, bh, js_distance, gini, boot_mean_ci

RNG = np.random.default_rng(42)
KEYS = ["platform", "conversation_id", "turn_index"]
RAREFY_N = 15       # turns sampled per user for breadth comparisons or number of conversations when relevant 
RAREFY_REPS = 25
N_BOOT = 1000
N_PERM = 1000
TOP_N = 12


# Content Analysis

In [ ]:
import pandas as pd 

# load full turn data 
dataset = pd.read_csv(f'{input_path}/merged_turns.csv')
# load tasks 
turns_long_tasks = pd.read_csv(f'{input_path}/merged_tasks.csv')
# load topics
turns_long_topics = pd.read_csv(f'{input_path}/merged_topics.csv')

In [4]:
# eligible dataset 
turns = dataset.copy()
for c in KEYS:
    turns[c] = turns[c].astype(str)
turns["user_id"] = turns["user_id"].astype(str)
turns = turns.drop_duplicates(KEYS)
turn_users = turns[KEYS + ["user_id"]]

user_size = (turns.groupby(["platform", "user_id"])
             .agg(n_turns=("turn_index", "size"))
             .reset_index())

eligible = set(user_size.loc[user_size["n_turns"] >= 5, "user_id"])

# Build per user topics and tasks profiles

In [56]:
# Creating matrixes for the analysis, that normalize tasks and topics per users
# tasks are normalized to account for turn distribution per users, topics to account for conversations labels instead of turn-level
def user_profiles(labels, col, eligible_ids):
    """One row per user and category, with fractional credit per turn."""
    lab = labels.copy()
    for c in KEYS:
        lab[c] = lab[c].astype(str)
    lab = lab.dropna(subset=[col]).drop_duplicates(KEYS + [col])
    lab["w"] = 1.0 / lab.groupby(KEYS)[col].transform("size")
    lab = lab.merge(turn_users, on=KEYS, how="inner", validate="many_to_one")
    lab = lab[lab["user_id"].isin(eligible_ids)]

    prof = (lab.groupby(["platform", "user_id", col])["w"].sum()
            .reset_index(name="w"))
    #prof["share"] = prof["w"] / prof.groupby("user_id")["w"].transform("sum")
    prof["share"] = prof["w"] / prof.groupby(["platform", "user_id"])["w"].transform("sum")

    pairs = lab.assign(turn_key=lab["conversation_id"] + "#" + lab["turn_index"])
    pairs = pairs[["platform", "user_id", "turn_key", col]].rename(columns={col: "label"})
    return prof, pairs

def to_matrix(prof, col):
    m = prof.pivot_table(index=["platform", "user_id"], columns=col,
                         values="share", fill_value=0.0)
    m.columns = [str(c) for c in m.columns]
    return m

def user_profiles_conv(labels, col, eligible_ids):
    """One row per user and category. Distinct labels per conversation split one unit."""
    lab = labels.copy()
    for c in KEYS:
        lab[c] = lab[c].astype(str)
    lab = lab.dropna(subset=[col])
    lab = lab.merge(turn_users, on=KEYS, how="inner", validate="many_to_one")
    lab = lab[lab["user_id"].isin(eligible_ids)]

    conv = ["platform", "conversation_id", "user_id"]
    lab = lab.drop_duplicates(conv + [col])
    lab["w"] = 1.0 / lab.groupby(conv)[col].transform("size")

    prof = (lab.groupby(["platform", "user_id", col])["w"].sum()
            .reset_index(name="w"))
    prof["share"] = prof["w"] / prof.groupby(["platform", "user_id"])["w"].transform("sum")

    pairs = lab[["platform", "user_id", "conversation_id", col]].rename(
        columns={col: "label", "conversation_id": "turn_key"})
    return prof, pairs


task_prof, task_pairs = user_profiles(turns_long_tasks, "task_mapped", eligible)
topic_prof, topic_pairs = user_profiles_conv(turns_long_topics, "topic", eligible)

task_mat = to_matrix(task_prof, "task_mapped")
topic_mat = to_matrix(topic_prof, "topic")

# Diversity & Breadth

In [6]:
# draw a random sample of exactly N units from each user, compute breadth,
# entropy, and top-1 share on just that sample, repeat it 50 times, and average.
RAREFY_N = 15   # raise toward the smallest platform's median; check min_turns first

# every eligible user is included. we sample N units at randoma and repeat 50 times
def rarefied_diversity(pairs, k_total, name, n=RAREFY_N, reps=50):
    rows = []
    for (platform, uid), g in pairs.groupby(["platform", "user_id"], sort=False):
        codes, inv = np.unique(g["turn_key"].values, return_inverse=True)
        if codes.size < n:
            continue
        lab = g["label"].values
        w = g["w"].values if "w" in g else np.ones(len(g))
        br, en, t1 = [], [], []
        for _ in range(reps):
            take = np.isin(inv, RNG.choice(codes.size, n, replace=False))
            s = pd.Series(w[take]).groupby(lab[take]).sum()
            s = s / s.sum()
            br.append(s.size)
            en.append(norm_entropy(s.values, k_total))
            t1.append(s.max())
        rows.append((platform, uid, np.mean(br), np.mean(en), np.mean(t1)))
    return pd.DataFrame(rows, columns=["platform", "user_id",
                                       "breadth", "entropy", "top1_share"]).assign(domain=name)

task_div = rarefied_diversity(task_pairs, task_mat.shape[1], "task")
topic_div = rarefied_diversity(topic_pairs, topic_mat.shape[1], "topic")

In [7]:
# compare the diversity measures
def compare(d, var):
    groups = [g[var].dropna().values for _, g in d.groupby("platform")]
    H, p = kruskal(*groups)
    rows = []
    plats = sorted(d["platform"].unique())
    for i, a in enumerate(plats):
        for b in plats[i + 1:]:
            x = d.loc[d["platform"] == a, var].dropna()
            y = d.loc[d["platform"] == b, var].dropna()
            U, pv = mannwhitneyu(x, y, alternative="two-sided")
            rows.append({"a": a, "b": b, "p": pv,
                         "rank_biserial": 1 - 2 * U / (len(x) * len(y))})
    r = pd.DataFrame(rows)
    r["p_fdr"] = bh(r["p"])
    return H, p, r

for name, d in [("task", task_div), ("topic", topic_div)]:
    print(f"\n{name} (N={RAREFY_N})")
    print(d.groupby("platform")[["breadth", "entropy", "top1_share"]]
          .agg(["median", "mean", "size"]).round(3).to_string())
    for var in ["breadth", "entropy", "top1_share"]:
        H, p, r = compare(d, var)
        print(f"\n{var}: Kruskal H={H:.1f}, p={p:.3g}")
        print(r.round(3).to_string(index=False))


task (N=15)
         breadth             entropy             top1_share            
          median   mean size  median   mean size     median   mean size
platform                                                               
chatgpt     7.57  7.302  100   0.636  0.613  100      0.298  0.323  100
claude      6.94  6.778   99   0.596  0.573   99      0.337  0.370   99
deepseek    6.90  6.828   98   0.598  0.579   98      0.325  0.361   98
gemini      7.06  6.937   99   0.605  0.588   99      0.327  0.349   99
grok        6.51  6.231   98   0.570  0.542   98      0.350  0.390   98

breadth: Kruskal H=42.9, p=1.08e-08
       a        b     p  rank_biserial  p_fdr
 chatgpt   claude 0.005         -0.232  0.008
 chatgpt deepseek 0.001         -0.272  0.003
 chatgpt   gemini 0.017         -0.197  0.024
 chatgpt     grok 0.000         -0.557  0.000
  claude deepseek 0.953          0.005  0.953
  claude   gemini 0.473          0.059  0.525
  claude     grok 0.003         -0.246  0.006
deepse